# Graph Triage v0 — Strawman Baseline vs. Minimal Graph

Implements `SIMPLIFIED_PLAN.md` end to end:

fixed slice → sessionization → strawman kNN anomaly score → minimal process-file-parent
graph → SAGE embedding kNN score → shared metrics → holdout check.

The question: can a simple process-file-parent graph produce a useful unsupervised
session anomaly score compared with a simple strawman baseline? Either outcome is an
acceptable v0 result — the protocol just has to run end to end and be reproducible.
Labels (`red_team`) are used only at evaluation time.

In [1]:
import json
import os
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import HeteroConv, SAGEConv
from scipy.spatial import cKDTree
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

## Frozen configuration

Everything tunable lives here and is dumped into every slice manifest.
Changing any of these values is a new experiment version, not a tweak.

In [2]:
CONFIG = {
    "train_path": "Data/train-process_uber_summary.parquet",
    "test_path": "Data/test-process_uber_summary.parquet",
    "full_path": "Data/process_uber_summary.parquet",
    "rows": 100_000,                  # dev slice = chronological tail(rows)
    "inactivity_gap_minutes": 5,      # session split rules (frozen for v0)
    "max_session_minutes": 30,
    "rare_file_min_degree": 2,        # keep files touched by 2..15 processes
    "rare_file_max_degree": 15,
    "knn_k": 15,                      # score = mean distance to k nearest sessions
    "sage_seeds": (42, 43, 44),       # graph metrics reported as mean ± std over these
    "sage_hidden": 48,
    "sage_out": 48,
    "sage_epochs": 8,
    "max_pos_per_edge_type": 5_000,
    "review_budgets": (25, 50, 100, 250),
    "random_seed": 42,
}

## Data slice

Deterministic slice rule: parse timestamps, sort chronologically, drop duplicate
`pid_hash`, take the tail. The same rule is applied to dev, holdout, and full runs.

In [3]:
def load_slice(path, rows):
    df = pd.read_parquet(path)
    df["_time"] = pd.to_datetime(df["process_started"], errors="coerce", utc=True)
    df = (
        df[df["_time"].notna()]
        .sort_values("_time")
        .drop_duplicates("pid_hash", keep="first")
        .tail(rows)
        .reset_index(drop=True)
    )
    df.attrs["source_file"] = os.path.basename(path)
    return df

## Sessionization (frozen for v0)

Sessions are the evaluation unit. Group rows by identity (user name; rows with no
user fall back to host identity), then split a group into a new session after
5 minutes of inactivity or 30 minutes total duration. A session is malicious if any
process in it carries the `red_team` flag — used only at evaluation time.

In [4]:
def sessionize(df, config):
    gap = pd.Timedelta(minutes=config["inactivity_gap_minutes"])
    max_span = pd.Timedelta(minutes=config["max_session_minutes"])

    user = df["user_name"].astype("string")
    host = df["hostname"].astype("string")
    identity = user.fillna("host:" + host.fillna("<no host>"))

    sessions = []
    for _, group in df.groupby(identity, sort=False):
        current, start, prev = [], None, None
        for idx, t in zip(group.index, group["_time"]):
            if current and (t - prev > gap or t - start > max_span):
                sessions.append(current)
                current = []
            if not current:
                start = t
            current.append(idx)
            prev = t
        if current:
            sessions.append(current)
    return sorted(sessions, key=lambda s: s[0])


def session_labels(df, sessions):
    flags = (df["red_team"].fillna(0).astype(int) == 1).to_numpy()
    return np.array(["malicious" if flags[s].any() else "benign" for s in sessions])


def session_id_array(n_rows, sessions):
    sid = np.empty(n_rows, dtype=np.int64)
    for i, s in enumerate(sessions):
        sid[s] = i
    return sid

## Strawman baseline: `raw_session_stats_knn`

Session raw stats → StandardScaler → kNN distance anomaly score
(mean distance to the k=15 nearest sessions; higher = review sooner).
Stats are simple aggregates only: size, duration, mean/std of the numeric process
fields, unique process/file names, rare-file count, and parent-child edge count.
Rare-file degrees and the scaler are refit on whatever slice is being scored.

In [5]:
def rare_file_set(df, config):
    counts = df["filename"].dropna().value_counts()
    keep = (counts >= config["rare_file_min_degree"]) & (counts <= config["rare_file_max_degree"])
    return set(counts[keep].index)


def parent_child_edges(df):
    """(parent_row, child_row) pairs where both processes are in the slice."""
    row_of_pid = pd.Series(np.arange(len(df)), index=df["pid_hash"])
    parent_row = df["parent_pid_hash"].map(row_of_pid).to_numpy(dtype=float)
    child_row = np.flatnonzero(~np.isnan(parent_row))
    return np.column_stack([parent_row[child_row].astype(np.int64), child_row])


def build_session_stats(df, sessions, rare_files, pc_edges):
    sid = session_id_array(len(df), sessions)
    g = df.assign(_sid=sid).groupby("_sid")

    stats = pd.DataFrame({
        "session_size": g.size(),
        "duration_seconds": (g["_time"].max() - g["_time"].min()).dt.total_seconds(),
        "unique_process_names": g["process_name"].nunique(),
        "unique_filenames": g["filename"].nunique(),
    })

    is_rare = df["filename"].isin(rare_files).to_numpy()
    rare_per_session = df.loc[is_rare].groupby(sid[is_rare])["filename"].nunique()
    stats["rare_file_count"] = rare_per_session.reindex(stats.index, fill_value=0)

    within = sid[pc_edges[:, 0]] == sid[pc_edges[:, 1]]
    stats["parent_child_edges"] = np.bincount(
        sid[pc_edges[within, 1]], minlength=len(sessions)
    )

    # mean/std of the numeric process fields; red_team and label_* stay evaluation-only
    numeric_cols = [
        c for c in df.select_dtypes(include=[np.number]).columns
        if c != "red_team" and not c.startswith("label_")
    ]
    agg = g[numeric_cols].agg(["mean", "std"])
    agg.columns = [f"{col}_{stat}" for col, stat in agg.columns]

    stats = stats.join(agg).fillna(0.0)
    stats.insert(0, "session_id", stats.index.to_numpy())
    stats.insert(1, "user", g["user_name"].first().fillna(""))
    stats.insert(2, "host", g["hostname"].first().fillna(""))
    return stats.reset_index(drop=True)


def knn_anomaly_scores(X, k):
    X = np.asarray(X, dtype=np.float32)
    if len(X) <= 1:
        return np.zeros(len(X))
    qk = min(k + 1, len(X))
    dists, _ = cKDTree(X).query(X, k=qk, workers=-1)
    return dists[:, 1:].mean(axis=1)

## Minimal graph

The v0 default graph has exactly three relations — `parent_child`, `touches`,
`touched_by` — over process and file nodes, with rare-file filtering. No same-user,
same-host, or temporal edges. Node features are constant, so the learned embedding
sees process-level structure only; a win or loss against the strawman is then
interpretable as structure vs. aggregates.

In [6]:
def build_graph(df, rare_files, pc_edges):
    touched = df["filename"].isin(rare_files).to_numpy()
    files = pd.Categorical(df.loc[touched, "filename"])
    touch_ei = torch.tensor(
        np.vstack([np.flatnonzero(touched), files.codes.astype(np.int64)]),
        dtype=torch.long,
    )

    edges = {
        ("process", "parent_child", "process"): torch.tensor(pc_edges.T, dtype=torch.long),
        ("process", "touches", "file"): touch_ei,
        ("file", "touched_by", "process"): touch_ei.flip(0),
    }
    num_nodes = {"process": len(df), "file": len(files.categories)}
    x_dict = {ntype: torch.ones((n, 1)) for ntype, n in num_nodes.items()}
    return edges, x_dict, num_nodes

## Graph score: `v0_graph_sage_knn`

Self-supervised link prediction: a two-layer HeteroSAGE encoder is trained to score
real edges above randomly sampled fake ones (no labels). Process embeddings are
mean-pooled into session embeddings, and the session anomaly score is the same
kNN mean-distance used by the strawman. One encoder per seed; each seed is scored
independently.

In [7]:
class SmallHeteroSAGE(nn.Module):
    def __init__(self, edge_types, hidden, out):
        super().__init__()
        self.convs = nn.ModuleList([
            HeteroConv({et: SAGEConv((-1, -1), hidden) for et in edge_types}, aggr="sum"),
            HeteroConv({et: SAGEConv((-1, -1), out) for et in edge_types}, aggr="sum"),
        ])

    def forward(self, x_dict, edge_index_dict):
        z = self.convs[0](x_dict, edge_index_dict)
        z = {k: F.relu(v) for k, v in z.items()}
        return self.convs[1](z, edge_index_dict)


def train_sage(edges, x_dict, num_nodes, config, seed):
    torch.manual_seed(seed)
    sampler = torch.Generator().manual_seed(seed + 10_000)
    model = SmallHeteroSAGE(list(edges), config["sage_hidden"], config["sage_out"])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(config["sage_epochs"]):
        opt.zero_grad()
        z = model(x_dict, edges)
        losses = []
        for (src, _, dst), pos in edges.items():
            n_pos = min(config["max_pos_per_edge_type"], pos.shape[1])
            if n_pos == 0:
                continue
            perm = torch.randperm(pos.shape[1], generator=sampler)[:n_pos]
            neg = torch.stack([
                torch.randint(0, num_nodes[src], (n_pos,), generator=sampler),
                torch.randint(0, num_nodes[dst], (n_pos,), generator=sampler),
            ])
            pos_score = 5.0 * F.cosine_similarity(z[src][pos[0, perm]], z[dst][pos[1, perm]], dim=1)
            neg_score = 5.0 * F.cosine_similarity(z[src][neg[0]], z[dst][neg[1]], dim=1)
            losses.append(
                F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
                + F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score))
            )
        loss = torch.stack(losses).mean()
        loss.backward()
        opt.step()
        if epoch in (0, config["sage_epochs"] - 1):
            print(f"  sage seed={seed} epoch={epoch:02d} loss={loss.item():.4f}")

    model.eval()
    with torch.no_grad():
        return model(x_dict, edges)["process"].numpy()


def session_embeddings(process_z, sessions):
    X = np.vstack([process_z[s].mean(axis=0) for s in sessions])
    return StandardScaler().fit_transform(np.nan_to_num(X))

## Metrics

Co-primary: reviews-to-first-malicious (reported next to its random expectation
N/(m+1), never alone) and average precision. Secondary: found/recall/precision at
review budgets 25/50/100/250. Graph rows carry ± std across seeds.

In [8]:
def discovery_metrics(scores, labels, method, budgets):
    y = (np.asarray(labels) == "malicious").astype(int)
    order = np.argsort(np.asarray(scores, dtype=float))[::-1]
    hits_in_order = np.flatnonzero(y[order] == 1)
    row = {
        "method": method,
        "average_precision": average_precision_score(y, scores) if y.sum() else np.nan,
        "reviews_to_first_malicious": float(hits_in_order[0] + 1) if len(hits_in_order) else np.nan,
    }
    for b in budgets:
        n = min(b, len(y))
        found = int(y[order[:n]].sum())
        row[f"found_at_{b}"] = found
        row[f"recall_at_{b}"] = found / y.sum() if y.sum() else np.nan
        row[f"precision_at_{b}"] = found / n
    return row


def summarize_methods(labels, score_map, graph_scores_by_seed, config):
    budgets = config["review_budgets"]
    rows = [discovery_metrics(s, labels, name, budgets) for name, s in score_map.items()]

    per_seed = pd.DataFrame([
        discovery_metrics(s, labels, "v0_graph_sage_knn", budgets)
        for s in graph_scores_by_seed.values()
    ]).drop(columns="method")
    graph_row = {"method": "v0_graph_sage_knn", **per_seed.mean().to_dict()}
    std_cols = ["average_precision", "reviews_to_first_malicious"] + [f"recall_at_{b}" for b in budgets]
    graph_row.update({f"{c}_std": per_seed[c].std() for c in std_cols})
    rows.append(graph_row)

    out = pd.DataFrame(rows)
    n, m = len(labels), int((labels == "malicious").sum())
    out.insert(3, "random_expected_reviews_to_first", n / (m + 1))
    return out

## Pipeline entry point

One thin function: `run_minimal_discovery_pipeline(df, config) -> results`.
Fully unsupervised per slice — the scaler, rare-file degrees, and kNN neighborhoods
are refit on whatever slice is passed in; only the config and code are shared.

In [9]:
def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    except Exception:
        return "unknown"


def slice_manifest(df, sessions, labels, file_node_count, config):
    mal = (df["red_team"].fillna(0).astype(int) == 1).to_numpy()
    return pd.DataFrame([{
        "source_file": df.attrs.get("source_file", "unknown"),
        "git_commit": git_commit(),
        "config": json.dumps(config),
        "row_count": len(df),
        "start_time": df["_time"].min(),
        "end_time": df["_time"].max(),
        "n_sessions": len(sessions),
        "n_malicious_sessions": int((labels == "malicious").sum()),
        "malicious_users": df.loc[mal, "user_name"].nunique(),
        "malicious_hosts": df.loc[mal, "hostname"].nunique(),
        "malicious_user_names": ", ".join(sorted(df.loc[mal, "user_name"].dropna().unique())),
        "benign_users": df.loc[~mal, "user_name"].nunique(),
        "benign_hosts": df.loc[~mal, "hostname"].nunique(),
        "file_nodes_after_rare_filter": file_node_count,
    }])


def run_minimal_discovery_pipeline(df, config):
    sessions = sessionize(df, config)
    labels = session_labels(df, sessions)  # evaluation-only from here on
    rare_files = rare_file_set(df, config)
    pc_edges = parent_child_edges(df)

    # strawman
    session_table = build_session_stats(df, sessions, rare_files, pc_edges)
    feature_cols = [c for c in session_table.columns if c not in ("session_id", "user", "host")]
    X = StandardScaler().fit_transform(session_table[feature_cols].to_numpy(dtype=np.float64))
    baseline_scores = pd.DataFrame({
        "session_id": session_table["session_id"],
        "label": labels,
        "raw_session_stats_knn": knn_anomaly_scores(X, config["knn_k"]),
    })

    # graph
    edges, x_dict, num_nodes = build_graph(df, rare_files, pc_edges)
    graph_scores = pd.DataFrame({"session_id": session_table["session_id"], "label": labels})
    by_seed = {}
    for seed in config["sage_seeds"]:
        z = train_sage(edges, x_dict, num_nodes, config, seed)
        by_seed[seed] = knn_anomaly_scores(session_embeddings(z, sessions), config["knn_k"])
        graph_scores[f"knn_seed_{seed}"] = by_seed[seed]
    graph_scores["v0_graph_score"] = graph_scores[[f"knn_seed_{s}" for s in config["sage_seeds"]]].mean(axis=1)

    rng = np.random.default_rng(config["random_seed"])
    metric_summary = summarize_methods(
        labels,
        {
            "random": rng.random(len(labels)),
            "raw_session_stats_knn": baseline_scores["raw_session_stats_knn"].to_numpy(),
        },
        by_seed,
        config,
    )

    edge_summary = pd.DataFrame([{
        "process_nodes": num_nodes["process"],
        "file_nodes": num_nodes["file"],
        "parent_child_edges": len(pc_edges),
        "touches_edges": edges[("process", "touches", "file")].shape[1],
        "touched_by_edges": edges[("file", "touched_by", "process")].shape[1],
    }])

    return {
        "slice_manifest": slice_manifest(df, sessions, labels, num_nodes["file"], config),
        "session_table": session_table,
        "baseline_scores": baseline_scores,
        "graph_scores": graph_scores,
        "metric_summary": metric_summary,
        "edge_summary": edge_summary,
    }

## Dev slice run

In [10]:
dev_df = load_slice(CONFIG["train_path"], CONFIG["rows"])
dev = run_minimal_discovery_pipeline(dev_df, CONFIG)

artifacts = Path("artifacts/v0")
artifacts.mkdir(parents=True, exist_ok=True)
dev["slice_manifest"].to_csv(artifacts / "dev_slice_manifest.csv", index=False)
dev["metric_summary"].to_csv(artifacts / "dev_metric_summary.csv", index=False)

display(dev["slice_manifest"])
display(dev["edge_summary"])
display(dev["metric_summary"].round(4))

  sage seed=42 epoch=00 loss=2.6201
  sage seed=42 epoch=07 loss=1.9455
  sage seed=43 epoch=00 loss=2.7307
  sage seed=43 epoch=07 loss=2.1194
  sage seed=44 epoch=00 loss=2.6767
  sage seed=44 epoch=07 loss=1.9986


,source_file,git_commit,config,row_count,start_time,end_time,n_sessions,n_malicious_sessions,malicious_users,malicious_hosts,malicious_user_names,benign_users,benign_hosts,file_nodes_after_rare_filter
0,train-process_uber_summary.parquet,c2d7b3cf00d84aac4765aeca3d9de4d307c7484a,"{""train_path"": ""Data/train-process_uber_summar...",100000,2024-09-18 20:45:15.961034+00:00,2024-09-22 23:57:33.487951+00:00,1457,19,1,1,ACME\baduser3,7,5,14


,process_nodes,file_nodes,parent_child_edges,touches_edges,touched_by_edges
0,100000,14,2318,66,66


,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,...,precision_at_100,found_at_250,recall_at_250,precision_at_250,average_precision_std,reviews_to_first_malicious_std,recall_at_25_std,recall_at_50_std,recall_at_100_std,recall_at_250_std
0,random,0.0114,105.0000,72.85,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,2.0,0.1053,0.008,NaN,NaN,NaN,NaN,NaN,NaN
1,raw_session_stats_knn,0.0170,149.0000,72.85,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.2105,0.016,NaN,NaN,NaN,NaN,NaN,NaN
2,v0_graph_sage_knn,0.0130,862.3333,72.85,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0000,0.000,0.0,6.8069,0.0,0.0,0.0,0.0


In [11]:
context_cols = ["session_id", "user", "host", "session_size", "duration_seconds"]

top_25_by_baseline_score = (
    dev["baseline_scores"]
    .merge(dev["session_table"][context_cols], on="session_id")
    .sort_values("raw_session_stats_knn", ascending=False)
    .head(25)
)
top_25_by_graph_score = (
    dev["graph_scores"][["session_id", "label", "v0_graph_score"]]
    .merge(dev["session_table"][context_cols], on="session_id")
    .sort_values("v0_graph_score", ascending=False)
    .head(25)
)
display(top_25_by_baseline_score)
display(top_25_by_graph_score)

,session_id,label,raw_session_stats_knn,user,host,session_size,duration_seconds
421,421,benign,240.442706,ACME\ghostuser2,ACME-HH-CWQ,13,5.995357
571,571,benign,145.568971,ACME\ghostuser1,EC2AMAZ-R9HHULK,9,5.642433
382,382,benign,65.555021,ACME\ghostuser1,EC2AMAZ-R9HHULK,1,0.000000
1128,1128,benign,57.781916,ACME\ghostuser2,ACME-HH-CWQ,12,23.962980
759,759,benign,54.554370,NT AUTHORITY\SYSTEM,ACME-HH-CWQ,70,1449.496941
76,76,benign,54.242512,NT AUTHORITY\SYSTEM,ACME-HH-CWQ,22,1773.789711
600,600,benign,43.242870,NT AUTHORITY\NETWORK SERVICE,ACME-HH-DXJ,2,232.760439
238,238,benign,43.060380,ACME\ghostuser1,EC2AMAZ-R9HHULK,5,58.993650
1188,1188,benign,40.867794,ACME\ghostuser1,EC2AMAZ-R9HHULK,3,0.016642
762,762,benign,38.435644,NT AUTHORITY\NETWORK SERVICE,EC2AMAZ-R9HHULK,1,0.000000


,session_id,label,v0_graph_score,user,host,session_size,duration_seconds
972,972,benign,16.115004,ACME\ghostuser2,ACME-HH-CWQ,5,30.661798
649,649,benign,16.115004,ACME\ghostuser2,ACME-HH-CWQ,5,30.406109
571,571,benign,9.574574,ACME\ghostuser1,EC2AMAZ-R9HHULK,9,5.642433
1051,1051,benign,8.036125,ACME\ghostuser2,ACME-HH-CWQ,5,96.368994
915,915,benign,7.305233,ACME\ghostuser2,ACME-HH-CWQ,3,6.769381
1315,1315,benign,6.955389,ACME\ghostuser1,EC2AMAZ-R9HHULK,12,5.760307
1128,1128,benign,6.955389,ACME\ghostuser2,ACME-HH-CWQ,12,23.962980
421,421,benign,6.523708,ACME\ghostuser2,ACME-HH-CWQ,13,5.995357
453,453,benign,5.727342,ACME\ghostuser2,ACME-HH-CWQ,3,105.353304
1310,1310,benign,5.702430,,EC2AMAZ-R9HHULK,63,1641.077475


## Tests

Alignment, label isolation, determinism, and a random-score sanity control,
per the plan's test list.

In [12]:
# Alignment: sessions exactly partition the slice rows, and graph process node i is df row i.
dev_sessions = sessionize(dev_df, CONFIG)
covered = np.concatenate(dev_sessions)
assert dev_df["pid_hash"].is_unique
assert len(covered) == len(dev_df)
assert np.array_equal(np.sort(covered), np.arange(len(dev_df)))
assert len(dev["session_table"]) == len(dev_sessions)
print(f"alignment ok: {len(dev_sessions)} sessions partition {len(dev_df)} rows")

alignment ok: 1457 sessions partition 100000 rows


In [13]:
# Label isolation: labels never enter features or graph training.
# build_session_stats mentions red_team only to exclude it, so it is checked
# through its output columns; the graph-side functions must not reference it at all.
import inspect

assert not any(c == "red_team" or "red_team" in c or c.startswith("label_")
               for c in dev["session_table"].columns)
for fn in (sessionize, build_graph, train_sage, session_embeddings):
    assert "red_team" not in inspect.getsource(fn), fn.__name__
print("label isolation ok: red_team appears only in session_labels / manifest / metrics")

label isolation ok: red_team appears only in session_labels / manifest / metrics


In [14]:
# Determinism: a second full run with the same seeds reproduces manifest and metrics.
rerun = run_minimal_discovery_pipeline(dev_df, CONFIG)
pd.testing.assert_frame_equal(dev["slice_manifest"], rerun["slice_manifest"])
pd.testing.assert_frame_equal(dev["metric_summary"], rerun["metric_summary"], atol=1e-8, check_exact=False)
print("determinism ok: identical manifest, matching metrics")

  sage seed=42 epoch=00 loss=2.6201
  sage seed=42 epoch=07 loss=1.9455
  sage seed=43 epoch=00 loss=2.7307
  sage seed=43 epoch=07 loss=2.1194
  sage seed=44 epoch=00 loss=2.6767
  sage seed=44 epoch=07 loss=1.9986
determinism ok: identical manifest, matching metrics


In [15]:
# Sanity control: random ordering should need ~N/(m+1) reviews to hit the first malicious session.
labels = dev["baseline_scores"]["label"].to_numpy()
n, m = len(labels), int((labels == "malicious").sum())
rng = np.random.default_rng(0)
first_hits = [
    discovery_metrics(rng.random(n), labels, "random", CONFIG["review_budgets"])["reviews_to_first_malicious"]
    for _ in range(200)
]
print(f"random reviews-to-first over 200 draws: {np.mean(first_hits):.1f} (expected ~{n / (m + 1):.1f})")

random reviews-to-first over 200 draws: 70.8 (expected ~72.8)


## Holdout

Viability check first, recorded before any holdout scores are computed. The v0 claim
only holds if the dev-slice direction (graph vs. strawman on both co-primary metrics)
replicates on the untouched holdout.

In [16]:
holdout_df = load_slice(CONFIG["test_path"], CONFIG["rows"])
holdout_sessions = sessionize(holdout_df, CONFIG)
holdout_labels = session_labels(holdout_df, holdout_sessions)
mal_rows = (holdout_df["red_team"].fillna(0).astype(int) == 1).to_numpy()

holdout_viability = pd.DataFrame([{
    "source_file": holdout_df.attrs["source_file"],
    "row_count": len(holdout_df),
    "red_team_rows": int(mal_rows.sum()),
    "n_sessions": len(holdout_sessions),
    "malicious_sessions": int((holdout_labels == "malicious").sum()),
    "malicious_users": holdout_df.loc[mal_rows, "user_name"].nunique(),
    "malicious_hosts": holdout_df.loc[mal_rows, "hostname"].nunique(),
}])
holdout_viability.to_csv(artifacts / "holdout_viability.csv", index=False)
display(holdout_viability)

holdout_viable = holdout_viability["malicious_sessions"].iat[0] >= 1
if not holdout_viable:
    print("Holdout slice has no red-team sessions: redesign the confirmatory set "
          "(e.g. a chronologically earlier train slice) before looking at any scores.")

,source_file,row_count,red_team_rows,n_sessions,malicious_sessions,malicious_users,malicious_hosts
0,test-process_uber_summary.parquet,100000,15,1074,8,1,1


In [17]:
def graph_vs_strawman(metric_summary):
    m = metric_summary.set_index("method")
    graph, straw = m.loc["v0_graph_sage_knn"], m.loc["raw_session_stats_knn"]
    return {
        "average_precision": "graph" if graph["average_precision"] > straw["average_precision"] else "strawman",
        "reviews_to_first": "graph" if graph["reviews_to_first_malicious"] < straw["reviews_to_first_malicious"] else "strawman",
    }


if holdout_viable:
    holdout = run_minimal_discovery_pipeline(holdout_df, CONFIG)
    holdout["slice_manifest"].to_csv(artifacts / "holdout_slice_manifest.csv", index=False)
    holdout["metric_summary"].to_csv(artifacts / "holdout_metric_summary.csv", index=False)
    display(holdout["metric_summary"].round(4))

    dev_direction = graph_vs_strawman(dev["metric_summary"])
    holdout_direction = graph_vs_strawman(holdout["metric_summary"])
    print("dev direction:    ", dev_direction)
    print("holdout direction:", holdout_direction)
    print("direction replicates on holdout:", dev_direction == holdout_direction)

  sage seed=42 epoch=00 loss=2.6981
  sage seed=42 epoch=07 loss=1.9970
  sage seed=43 epoch=00 loss=2.7040
  sage seed=43 epoch=07 loss=2.1629
  sage seed=44 epoch=00 loss=2.7135
  sage seed=44 epoch=07 loss=2.0621


,method,average_precision,reviews_to_first_malicious,random_expected_reviews_to_first,found_at_25,recall_at_25,precision_at_25,found_at_50,recall_at_50,precision_at_50,...,precision_at_100,found_at_250,recall_at_250,precision_at_250,average_precision_std,reviews_to_first_malicious_std,recall_at_25_std,recall_at_50_std,recall_at_100_std,recall_at_250_std
0,random,0.0103,132.0000,119.3333,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,1.0,0.125,0.004,NaN,NaN,NaN,NaN,NaN,NaN
1,raw_session_stats_knn,0.0187,68.0000,119.3333,0.0,0.0,0.0,0.0,0.0,0.0,...,0.02,4.0,0.500,0.016,NaN,NaN,NaN,NaN,NaN,NaN
2,v0_graph_sage_knn,0.0074,631.3333,119.3333,0.0,0.0,0.0,0.0,0.0,0.0,...,0.00,0.0,0.000,0.000,0.0,8.5049,0.0,0.0,0.0,0.0


dev direction:     {'average_precision': 'strawman', 'reviews_to_first': 'strawman'}
holdout direction: {'average_precision': 'strawman', 'reviews_to_first': 'strawman'}
direction replicates on holdout: True


## Full-set check

Run only after the dev/holdout protocol works, to test stability and runtime —
not to tune v0.

In [18]:
# full_df = load_slice(CONFIG["full_path"], rows=10**9)
# full = run_minimal_discovery_pipeline(full_df, CONFIG)
# display(full["metric_summary"].round(4))